# CS152 Final Project — Reversi AI
**Human (Black ⬤) vs AI (White ○)**

Run all cells in order. Click a **yellow-highlighted** square to place your piece.

> **Difficulty**: search depth is set by `search_depth(4)` in `reversi.pl`. Change the integer to adjust strength.

In [ ]:
# !pip install pyswip ipywidgets
# !apt-get install -y swi-prolog

In [ ]:
from reversi_ui import ReversiGame
game = ReversiGame()
game.show()

---
## Analysis: Search Time vs Depth

Benchmarks best_move at depths 1-5 from two board positions.

In [ ]:
import time, re
import matplotlib.pyplot as plt
import numpy as np
from pyswip import Prolog

pl = Prolog()
pl.consult('reversi.pl')

def board_str(b): return '[' + ','.join(b) + ']'
def parse_moves(raw):
    return [(int(re.findall(r'\\d+', str(m))[0]),
             int(re.findall(r'\\d+', str(m))[1])) for m in raw]

# Build initial board
board = [str(x) for x in list(pl.query('initial_board(B)'))[0]['B']]
bl = board_str(board)

# Build mid-game board (3 moves in)
b2 = [str(x) for x in list(pl.query(f'apply_move({bl},k,2,3,NB)'))[0]['NB']]
b3 = [str(x) for x in list(pl.query(f'apply_move({board_str(b2)},w,2,4,NB)'))[0]['NB']]
b4 = [str(x) for x in list(pl.query(f'apply_move({board_str(b3)},k,5,3,NB)'))[0]['NB']]

depths = [1, 2, 3, 4, 5]
t_open, t_mid = [], []
for d in depths:
    list(pl.query('retract(search_depth(_))')); pl.assertz(f'search_depth({d})')
    t = time.time(); list(pl.query(f'best_move({bl},k,_,_,_)')); t_open.append(time.time()-t)
    t = time.time(); list(pl.query(f'best_move({board_str(b4)},w,_,_,_)')); t_mid.append(time.time()-t)
    print(f'Depth {d}: opening={t_open[-1]:.3f}s  mid={t_mid[-1]:.3f}s')

list(pl.query('retract(search_depth(_))')); pl.assertz('search_depth(4)')

x = np.arange(len(depths))
fig, ax = plt.subplots(figsize=(8,4))
ax.bar(x-0.18, t_open, 0.35, label='Opening', color='#4a90d9')
ax.bar(x+0.18, t_mid,  0.35, label='Mid-game', color='#e07b39')
ax.set_xticks(x); ax.set_xticklabels(depths)
ax.set_xlabel('Search Depth'); ax.set_ylabel('Time (s)')
ax.set_title('Alpha-Beta Search Time vs Depth')
ax.set_yscale('log'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

---
## Analysis: Positional Weight Heatmap

In [ ]:
import numpy as np, matplotlib.pyplot as plt

corner = [(0,0),(0,7),(7,0),(7,7)]
xsq    = [(1,1),(1,6),(6,1),(6,6)]
csq    = [(0,1),(1,0),(0,6),(6,0),(7,1),(1,7),(7,6),(6,7)]
grid   = np.zeros((8,8))
for r in range(8):
    for c in range(8):
        if (r,c) in corner:          grid[r,c] = 100
        elif (r,c) in xsq:           grid[r,c] = -20
        elif (r,c) in csq:           grid[r,c] = -10
        elif r in (0,7) or c in (0,7): grid[r,c] = 10
        elif r in (1,6) or c in (1,6): grid[r,c] = 5
        else:                         grid[r,c] = 3

fig, ax = plt.subplots(figsize=(5,5))
im = ax.imshow(grid, cmap='RdYlGn', vmin=-25, vmax=105)
for r in range(8):
    for c in range(8):
        ax.text(c, r, str(int(grid[r,c])), ha='center', va='center', fontsize=9,
                color='white' if abs(grid[r,c])>=50 else 'black', fontweight='bold')
ax.set_title('Positional Weight Table'); fig.colorbar(im)
plt.tight_layout(); plt.show()